In [271]:
%run CertificateClassGroupLean2.0.ipynb
load('IrreducibilityLeanProofWriter.sage')

In [1]:
R.<X>=PolynomialRing(ZZ)

In [3]:
K.<a> = NumberField(X^5 + 10*X^3 - 10*X^2 - 15*X - 18)

In [63]:
B = [1, a, a^2, 1/2*a^3 - 1/2*a, 1/24*a^4 - 1/8*a^3 - 5/24*a^2 + 5/24*a - 1/4]

In [5]:
O = K.ring_of_integers()

In [223]:
F = factor(O.ideal(3))

In [230]:
list(F[1][0].gens())

[3, -1/8*a^4 + 3/8*a^3 - 11/8*a^2 + 35/8*a - 1/4]

In [166]:
#ideal_gens = [3, -5/24*a^4 + 1/8*a^3 - 47/24*a^2 + 59/24*a + 13/4]

In [170]:
#Computes a primitive element for the field O/I. 

def RandomPrimitiveElementQuot(K,ideal_gens,p):
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    a = K.gen()
    flag = 0 
    cont = 0 
    while flag == 0 and cont < 100:
        g = O.random_element()
        cont = cont + 1
        poly = g.minpoly()
        F = factor(GF(p)[x](g.minpoly()))
        for f in F:
            if p ** f[0].degree() == I.norm() and ((ZZ[X](f[0])(g)) in I):
                flag = 1
                return g, f[0]

In [208]:
#Writes a proof of primality for the ideal given by its generators (as an O-module). The name of the ideal 
# is index by the prime above it and the label. So ideal_namepNlabel. 

def PrimalityCertificate (K, B, ideal_gens, p, ideal_name, label, name_prime_proof):
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    N = I.norm()
    ideal_name_full = ideal_name + str(p) + 'N' + str(label)
    gensc = [elems_to_basis([x], B).list() for x in ideal_gens]
    print(f'def {ideal_name_full} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
    print('')
    span_label = 'SI'+str(p)+'N'+str(label)
    w = IdealEqSpanCertificateLean(K, ideal_gens, B, span_label, ideal_name_full)
    ik, jk = FindNzEntry(w) 
    print('')
    print(f'lemma N{ideal_name_full} : Nat.card (O ⧸ {ideal_name_full}) = {N} := ')
    print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ {span_label})")            
    print('')
    
    if N == p: 
        print(f'lemma isPrime{ideal_name_full} : Ideal.IsPrime {ideal_name_full} := prime_ideal_of_norm_prime {name_prime_proof} _ N{ideal_name_full}')
    else: 
        g, f = RandomPrimitiveElementQuot(K,ideal_gens,p)
        a = list((elems_to_basis([g],B)).transpose()[0])
        c = list((elems_to_basis([ZZ[X](f)(g)],B)).transpose()[0])
        IdealMemZLean(K, B,ideal_gens, c, 'MemC'+ideal_name_full, ideal_name_full, span_label)
        print('')
        CertificateIrrModpFFP(GF(p)[X](f),p,label,sys.stdout)
        print(f'''
def P{ideal_name_full} : PrimeIdeal O {p} where 
  r := {len(B)}
  n := {f.degree()}
  hpos := by decide
  TT := timesTableO
  T := Table
  heq := timesTableT_eq_Table
  I := {ideal_name_full}
  hcard := N{ideal_name_full}
  P := {list(f)}
  f := ofList {list(f)}
  hfeq := by decide
  hirr := irreducible_ofList_ofCertificateIrreducibleZModOfList' P{p}P{label}
  hneq := by decide
  hlen := by decide
  c := !{c}
  a := !{a}
  z := ![1,0,0,0,0]
  hBz := B_one_repr
  hpol := by decide
  hcmem := mem_of_certificate O ℤ _ _ _ _ MemC{ideal_name_full}
  hpmem := by 
    show  _ ∈ {ideal_name_full}.carrier
    rw [ideal_eq_of_IdealEqSpanCertificate O ℤ {span_label}]
    apply Submodule.subset_span
    use 0 ; dsimp ; congr ; norm_num
        
        ''')
        print(f'lemma isPrime{ideal_name_full} : Ideal.IsPrime {ideal_name_full} := PrimeIdeal_isPrime P{ideal_name_full}')
        
        

    
   
   # g, f = RandomPrimitiveElementQuot(K,ideal_gens,p)
    

In [237]:
def PrimesBelowGens(K,p):
    O = K.ring_of_integers() 
    F = factor(O.ideal(p))
    ideal_gens = [[[], F[i][1]] for i in range(len(F))]
    for i in range(len(F)):
        ideal_gens[i][0] = list(F[i][0].gens_reduced())
    return ideal_gens
    

In [276]:
p = 13
F = PrimesBelowGens(K,p)
l = len(F)

for i in range(l):
    if l > 1 : 
        PrimalityCertificate(K, B, F[i][0], p, 'I',i,'hp' + str(p) + '.out')
    else: 
        PrimalityCertificate(K, B, F[i][0], p, 'I','','hp' + str(p) + '.out')
    
ideal_gens = [F[i][0] for i in range(l)]
exp = [F[i][1] for i in range(l)]

IteratedMulLean(K, B, ideal_gens, exp, 'rfl', 'rfl', 'I' + str(p) + 'N')

ideal_names_list = []

for i in range(l):
    if l > 1 :
        ideal_names_list = ideal_names_list + F[i][1] * ['I' + str(p) + 'N' + str(i)]
    else :
        ideal_names_list = ideal_names_list + F[i][1] * ['I' + str(p) + 'N']

name_ideals_rep = "[" + ", ".join(ideal_names_list) + "]"



print(f'''def PBC{p} : PrimesBelowPCertificate {p} !{name_ideals_rep} where 
  Ip := by 
    intro i 
    fin_cases i ''')
for i in range(len(ideal_names_list)): 
    print('    exact isPrime' + ideal_names_list[i])

if len(ideal_names_list) >= 2 : 
    print(f'''  hPprod := by 
    simp only [Fin.prod_univ_succ, Fin.prod_univ_zero, mul_one, ← Ideal.mul_assoc]
    dsimp
    rw [ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ MulI{p}N{len(ideal_names_list)-2}, Set.range_unique]
    dsimp ; congr 
    erw [B_int_repr]
    rfl''')  
else : 
    print(f'''  hPprod := by 
    simp only [Fin.prod_univ_succ, Fin.prod_univ_zero, mul_one, ← Ideal.mul_assoc]
    dsimp
    unfold I{p}N
    rw [Set.range_unique]
    dsimp ; congr 
    erw [B_int_repr]
    rfl''')  
    
  


def I13N : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (![![13, 0, 0, 0, 0]] i)))

def SI13N: IdealEqSpanCertificate O ℤ timesTableO I13N
 ![![13, 0, 0, 0, 0]] 
 ![![13, 0, 0, 0, 0], ![0, 13, 0, 0, 0], ![0, 0, 13, 0, 0], ![0, 0, 0, 13, 0], ![0, 0, 0, 0, 13]] where
  T := Table 
  heq := timesTableT_eq_Table
  hieq := rfl 
  M :=![![![13, 0, 0, 0, 0], ![0, 13, 0, 0, 0], ![0, 0, 13, 0, 0], ![0, 0, 0, 13, 0], ![0, 0, 0, 0, 13]]]
  hmulB := by decide
  f := ![![![1, 0, 0, 0, 0]], ![![0, 1, 0, 0, 0]], ![![0, 0, 1, 0, 0]], ![![0, 0, 0, 1, 0]], ![![0, 0, 0, 0, 1]]]
  g := ![![![1, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 1]]]
  hle1 := by decide
  hle2 := by decide

lemma NI13N : Nat.card (O ⧸ I13N) = 371293 := 
 ideal_norm_eq_prod' B _ _ (by decide) 0 0 (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ SI13N)

def MemCI13N : IdealMemCertificate O ℤ B I13N
  ![![13, 0, 0, 0, 0], ![0, 13, 0, 0, 0], ![0, 0, 13, 0, 0], ![0, 0, 0, 13, 0], ![

In [218]:
set_random_seed(10)
PrimalityCertificate(K, B, [3, a + 1], 3, 'I',2,'hp3.out')

def I3N2 : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (![![3, 0, 0, 0, 0], ![1, 1, 0, 0, 0]] i)))

def SI3N2: IdealEqSpanCertificate O ℤ timesTableO I3N2
 ![![3, 0, 0, 0, 0], ![1, 1, 0, 0, 0]] 
 ![![3, 0, 0, 0, 0], ![1, 1, 0, 0, 0], ![2, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 1]] where
  T := Table 
  heq := timesTableT_eq_Table
  hieq := rfl 
  M :=![![![3, 0, 0, 0, 0], ![0, 3, 0, 0, 0], ![0, 0, 3, 0, 0], ![0, 0, 0, 3, 0], ![0, 0, 0, 0, 3]], ![![1, 1, 0, 0, 0], ![0, 1, 1, 0, 0], ![0, 1, 1, 2, 0], ![3, -1, 2, 4, 12], ![0, 0, 0, -2, -2]]]
  hmulB := by decide
  f := ![![![0, -1, 0, 0, 0], ![3, 0, 0, 0, 0]], ![![0, 0, 0, 0, 0], ![1, 0, 0, 0, 0]], ![![0, -1, 0, 0, 0], ![2, 1, 0, 0, 0]], ![![0, -1, -1, -1, 0], ![0, 1, 2, 0, 0]], ![![0, 2, 2, 4, -1], ![0, 2, -8, 0, -2]]]
  g := ![![![1, 0, 0, 0, 0], ![-1, 3, 0, 0, 0], ![-2, 0, 3, 0, 0], ![0, 0, 0, 3, 0], ![0, 0, 0, 0, 3]], ![![0, 1, 0, 0, 0], ![-1, 1, 1, 0, 0], ![-1, 1, 1, 2, 0], ![0, -1, 2, 4, 12], ![0, 0, 0, -2, -2]]]

In [220]:
#def IteratedMulLean (K, B, ideal_gens, ideal_pows, proof1_name, proof2_name, ideal_name):

IteratedMulLean(K, B, [[3,a], [3, -5/24*a^4 + 1/8*a^3 - 47/24*a^2 + 59/24*a + 13/4],[3,a + 1]], [1,1,1], 'rfl', 'rfl', 'I3N')

def MulI3N0 : IdealMulEqCertificate O ℤ timesTableO (I3N0) I3N1
  ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0]] ![![3, 0, 0, 0, 0], ![2, 3, -3, -1, -5]]
  ![![3, 0, 0, 0, 0], ![0, -6, 4, 1, 6]] where
 T := Table
 heq := timesTableT_eq_Table
 hI1 := rfl
 hI2 := rfl
 M :=  ![![![9, 0, 0, 0, 0], ![6, 9, -9, -3, -15]], ![![0, 3, 0, 0, 0], ![-3, 0, 1, 1, 3]]]
 hmul := by decide
 f :=  ![![![![-2, -1, 1, 0, 2], ![1, -3, 0, 0, 2]], ![![2, 1, 0, 0, 0], ![0, 0, 0, 0, 0]]], ![![![-7, 0, 0, -2, 1], ![0, -15, 0, 0, 10]], ![![8, 5, 0, 0, 0], ![4, 0, 0, 0, 0]]]]
 g :=  ![![![![3, 0, 0, 0, 0], ![0, 0, 0, 0, 0]], ![![2, 3, -3, -1, -5], ![0, 0, 0, 0, 0]]], ![![![0, 1, 0, 0, 0], ![0, 0, 0, 0, 0]], ![![-1, 2, -1, 0, -1], ![1, 0, 0, 0, 0]]]]
 hle1 := by decide
 hle2 := by decide

def MulI3N1 : IdealMulEqCertificate O ℤ timesTableO (I3N0*I3N1) I3N2
  ![![3, 0, 0, 0, 0], ![0, -6, 4, 1, 6]] ![![3, 0, 0, 0, 0], ![1, 1, 0, 0, 0]]
  ![![3, 0, 0, 0, 0]] where
 T := Table
 heq := timesTableT_eq_Table
 hI1 := ideal_eq_mul

(3,)

In [26]:
list_ideal_gens = [3,a]
ideal_name = 'I'

gensc = [elems_to_basis([x], B).list() for x in list_ideal_gens]
print(f'def {ideal_name} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')

w = IdealEqSpanCertificateLean(K, list_ideal_gens, B, 'A', ideal_name)
ik, jk = FindNzEntry(w) 
print('')
print(f'lemma N : Nat.card (O ⧸ {ideal_name}{i}) = {O.ideal(list_ideal_gens).norm()} := ')
print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ A)")            
print('')

def I : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0]] i)))
def A: IdealEqSpanCertificate O ℤ timesTableO I
 ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0]] 
 ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 3]] where
  T := Table 
  heq := timesTableT_eq_Table
  hieq := rfl 
  M :=![![![3, 0, 0, 0, 0], ![0, 3, 0, 0, 0], ![0, 0, 3, 0, 0], ![0, 0, 0, 3, 0], ![0, 0, 0, 0, 3]], ![![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 1, 0, 2, 0], ![3, -1, 2, 3, 12], ![0, 0, 0, -2, -3]]]
  hmulB := by decide
  f := ![![![1, 0, 0, 0, 0], ![0, 0, 0, 0, 0]], ![![0, 0, 0, 0, 0], ![1, 0, 0, 0, 0]], ![![0, 0, 0, 0, 0], ![0, 1, 0, 0, 0]], ![![0, -1, 0, -1, 0], ![1, 0, 2, 0, 0]], ![![0, 0, 0, 0, 1], ![0, 0, 0, 0, 0]]]
  g := ![![![1, 0, 0, 0, 0], ![0, 3, 0, 0, 0], ![0, 0, 3, 0, 0], ![0, 0, 0, 3, 0], ![0, 0, 0, 0, 1]], ![![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 1, 0, 2, 0], ![1, -1, 2, 3, 4], ![0, 0, 0, -2, -1]]]
  hle1 := by decide
 

In [139]:
set_random_seed(10)
g, f = RandomPrimitiveElementQuot(K, list_ideal_gens, 3)

In [142]:
f

x^2 + 2*x + 2

In [188]:
list((elems_to_basis([g],B)).transpose()[0])

[-17, 4, -15, -20, -68]

In [141]:
list((elems_to_basis([ZZ[X](f)(g)],B)).transpose())

[(5583, -3658, 3709, 4702, 22416)]

In [75]:
ZZ[X](f)(g) in I

False

In [143]:
IdealMemZLean(K, B,list_ideal_gens, [5583, -3658, 3709, 4702, 22416], 'MemC', 'I', 'A')

def MemC : IdealMemCertificate O ℤ B I
  ![![3, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 3]] ![5583, -3658, 3709, 4702, 22416] where
 hieq := ideal_eq_of_IdealEqSpanCertificate _ _ A
 g := ![1861, -3658, 3709, 4702, 7472]
 hmem := by decide


In [189]:
import sys
CertificateIrrModpFFP(GF(3)[X](f),3,0,sys.stdout)

def P3P0 : CertificateIrreducibleZModOfList' 3 2 2 1 [2, 2, 1] where
 m := 1
 P := ![2]
 exp := ![1] 
 hneq := by decide
 hP := by decide
 hlen := by decide
 htr := by decide
 bit := ![1, 1]
 hbits := by decide
 h := ![[0, 1], [1, 2], [0, 1]]
 g := ![![[1, 1]],![[2, 2]]]
 h' := ![![[1, 2], [0, 1]],![[0, 1], [1, 2]]]
 hs := by decide
 hz := by decide
 hmul := by decide
 a := ![[], [1]]
 b := ![[], [2, 2]]
 hhz := by decide
 hhn := by decide
 hgcd := by decide


In [65]:
len(B)

5